<a href="https://colab.research.google.com/github/siva123h/NNDL/blob/main/Transformer_Based_classifier_for_Email_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Install required libraries (run once)
# !pip install transformers datasets torch scikit-learn

import torch
from transformers import AutoTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset
from sklearn.model_selection import train_test_split

# -----------------------------
# 1. Sample Email Dataset
# -----------------------------
emails = [
    "Win a free lottery now",
    "Meeting scheduled at 10 AM",
    "Claim your free prize today",
    "Project deadline tomorrow",
    "Limited offer just for you",
    "Let's discuss the report"
]

labels = [1, 0, 1, 0, 1, 0]  # 1 = Spam, 0 = Not Spam

# Train-test split
train_texts, test_texts, train_labels, test_labels = train_test_split(
    emails, labels, test_size=0.2, random_state=42
)

# -----------------------------
# 2. Load Tokenizer
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize(example):
    return tokenizer(example['text'], padding='max_length', truncation=True)

# -----------------------------
# 3. Create Dataset (FIXED)
# -----------------------------
train_dataset = Dataset.from_dict({'text': train_texts, 'labels': train_labels})
test_dataset = Dataset.from_dict({'text': test_texts, 'labels': test_labels})

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# -----------------------------
# 4. Load Model
# -----------------------------
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

# -----------------------------
# 5. Training Setup (FIXED)
# -----------------------------
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

# -----------------------------
# 6. Train Model
# -----------------------------
trainer.train()

# -----------------------------
# 7. Prediction Function (FIXED)
# -----------------------------
def predict_email(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predicted_class = torch.argmax(logits, dim=1).item()

    return "Spam" if predicted_class == 1 else "Not Spam"

# -----------------------------
# 8. Test Prediction
# -----------------------------
test_email = "Congratulations! You won a free gift"
print("Email:", test_email)
print("Prediction:", predict_email(test_email))

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packag

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Email: Congratulations! You won a free gift
Prediction: Spam
